In [1]:
### Cu 003 processing ###


#%% load the packages
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

import defdap.hrdic as hrdic
import defdap.ebsd as ebsd
import defdap.experiment as experiment

from pathlib import Path

import copy 
import pandas as pd
import datetime

from scipy.signal import find_peaks
from scipy.interpolate import griddata
from scipy.linalg import lstsq
from scipy.ndimage import median_filter

import os

# get dictools stuff 
import sys
# sys.path.append("c:/work/hrdic-tools/")
# import dictools

plt.rcParams['svg.fonttype'] = 'none'

%matplotlib qt

In [2]:
def lsm_read(lsm_file):
    # for reading in lsm output from ZEISS Confomap

    df = pd.read_csv(lsm_file,names=['x','y','z'])

    x = np.asarray(df['x'])
    y = np.asarray(df['y'])
    z = np.asarray(df['z'])

    # calculate shape - this must be done on the raw data 
    x0 = np.nanmin(x)
    x1 = np.nanmax(x)
    y0 = np.nanmin(y)
    y1 = np.nanmax(y)

    x_size = x1 - x0
    y_size = y1 - y0

    # calculate step size 
    x_step = np.round(np.min(np.abs(np.diff(x))),4)
    y_step = np.round(np.max(np.abs(np.diff(y))),4)


    # create new grid to interpolate data onto
    xg,yg = np.meshgrid(np.arange(x0,x1,x_step),np.arange(y0,y1,y_step))



    # remove the weird way that ConfoMaps saves non-measured points
    x = x[z !='***']
    y = y[z !='***']
    z = z[z !='***']


    # interpolate onto grid to produced gridded data
    zg = griddata(np.asarray([x,y]).T,z,(xg,yg),method='nearest')

    # flip up down for zg
    zg = np.flipud(zg)

    return xg, yg, zg, x_step

def resample_gridded_data(xg,yg,zg,new_step):

    # flatten arrays - we could probably use RegularGridInterpolator but this works for now
    x = xg.flatten()
    y = yg.flatten()
    z = zg.flatten()

    # calculate shape - this must be done on the raw data
    x0 = np.nanmin(x)
    x1 = np.nanmax(x)
    y0 = np.nanmin(y)
    y1 = np.nanmax(y)

    x_size = x1 - x0
    y_size = y1 - y0

    # calculate step size 
    x_step = np.round(np.min(np.abs(np.diff(x))),4)
    y_step = np.round(np.max(np.abs(np.diff(y))),4)

    # create new grid to interpolate data onto
    xg_new,yg_new = np.meshgrid(np.arange(x0,x1,new_step),np.arange(y0,y1,new_step))

    # interpolate onto grid to produced gridded data
    zg_new = griddata(np.asarray([x,y]).T,z,(xg_new,yg_new),method='nearest')

    return xg_new, yg_new, zg_new


In [3]:

# path to LSM file 
lsm_file = './LSM/raw_surface.txt'

# import data
xg,yg,zg,lsm_step = lsm_read(lsm_file)




# processing to get data into useful form 
# crop the rubbish data from edges 
xL = 800
xR = 800
yT = 800
yB = 800

# if we want to nanify the deleted data
# zg[:,:xL] = np.nan
# zg[:,-xR:] = np.nan

# zg[:yT,:] = np.nan
# zg[-yB:,:] = np.nan

# plt.figure()
# plt.imshow(zg)

zg = zg[yT:-yB,xL:-xR]
xg = xg[yT:-yB,xL:-xR]
yg = yg[yT:-yB,xL:-xR]
# plt.figure()
# plt.imshow(zg)



# best-fit linear plane
A = np.c_[xg.flatten(),yg.flatten(), np.ones(xg.flatten().shape[0])]

C,_,_,_ = lstsq(A, zg.flatten())    # coefficients
    
# fitted plane
zg_fit = C[0]*xg + C[1]*yg + C[2]

# plt.figure()
fig,ax = plt.subplots(1,3)
ax[0].imshow(zg)
ax[0].set_title('Raw surface')
ax[1].imshow(zg_fit)
ax[1].set_title('Fitted flat plane')
ax[2].imshow(zg - zg_fit)
ax[2].set_title('Corrected')

plt.tight_layout()

# corrected surface
zg_flat = zg - zg_fit

# set lowest point on map to zero 
zg_flat = zg_flat - zg_flat.min()

In [4]:
# current dataset is overkill for DIC
# resample to DIC step size 

dic_step_px = 10
dic_px_size = 20/2048
dic_step = dic_step_px*dic_px_size # microns 



xg_new, yg_new, zg_new = resample_gridded_data(xg,yg,zg_flat,dic_step)



In [5]:
# median filter to smooth out bumps from speckle pattern

# filter kernel size
k_size = 15

zg_filtered = median_filter(zg_new,k_size)



In [6]:
fig,ax = plt.subplots()
surf = ax.imshow(zg_filtered,cmap='terrain',vmin=0,vmax=2)
bar = plt.colorbar(surf)
bar.set_label('Altitude / μm')

In [ ]:
# surface map 
# fig,ax = plt.subplots(subplot_kw={"projection":"3d"})

# rcount = 1000

# surf = ax.plot_surface(xg_new, yg_new, zg_new,rcount=rcount, ccount=rcount,cmap='terrain')
# ax.set_box_aspect((np.ptp(xg_new),np.ptp(yg_new),np.ptp(zg_new)))
# ax.set_zlim(0,2)
# fig.colorbar(surf)



: 